In [1]:
import pandas as pd
import numpy as np
import requests
import json
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

In [2]:
df = pd.read_csv("../Dashboards/powerbi_data/provider_dashboard_data.csv")

print("Dataset loaded successfully")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully
Shape: (1296739, 16)


,rndrng_npi,rndrng_prvdr_last_org_name,rndrng_prvdr_first_name,rndrng_prvdr_type,rndrng_prvdr_state_abrvtn,tot_benes,tot_srvcs,tot_sbmtd_chrg,tot_mdcr_pymt_amt,payment_per_beneficiary,services_per_beneficiary,charge_to_payment_ratio,payment_vs_state_avg,payment_vs_specialty_avg,fraud_risk_score,risk_category
0,1003000126,Enkeshafi,Ardalan,Internal Medicine,MD,328,399.0,202783.88,35325.46,107.699573,1.216463,5.740446,0.277118,0.417696,3.852433,Low Risk
1,1003000134,Cibull,Thomas,Pathology,IL,912,2023.0,302581.00,50178.11,55.019857,2.218202,6.030139,0.528989,0.556196,8.821533,Low Risk
2,1003000142,Khalil,Rashid,Anesthesiology,OH,316,1369.0,381301.00,101577.94,321.449177,4.332278,3.753778,1.680158,3.070757,11.436628,Low Risk
3,1003000423,Velotta,Jennifer,Obstetrics & Gynecology,OH,75,140.0,22015.00,7134.32,95.124267,1.866667,3.085788,0.118006,0.400652,1.517909,Low Risk
4,1003000480,Rothchild,Kevin,General Surgery,CO,101,136.0,120420.00,18766.80,185.809901,1.346535,6.416651,0.242330,0.252014,2.780082,Low Risk


In [3]:
required_columns = [
    "rndrng_npi",
    "rndrng_prvdr_last_org_name",
    "rndrng_prvdr_first_name",
    "rndrng_prvdr_type",
    "rndrng_prvdr_state_abrvtn",
    "tot_benes",
    "tot_srvcs",
    "tot_mdcr_pymt_amt",
    "payment_per_beneficiary",
    "services_per_beneficiary",
    "charge_to_payment_ratio",
    "payment_vs_state_avg",
    "payment_vs_specialty_avg",
    "fraud_risk_score",
    "risk_category"
]

missing_cols = [col for col in required_columns if col not in df.columns]

if missing_cols:
    print("Missing columns:", missing_cols)
else:
    print("All required columns are available.")

All required columns are available.


In [4]:
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "llama3.1"

def ask_llama(prompt):
    payload = {
        "model": MODEL,
        "prompt": prompt,
        "stream": False
    }

    response = requests.post(OLLAMA_URL, json=payload)
    response.raise_for_status()

    return response.json()["response"]

print("Ollama connection function created.")

Ollama connection function created.


In [5]:
print(ask_llama("Explain healthcare fraud in two sentences."))

Healthcare fraud occurs when an individual or organization intentionally misrepresents or conceals information to obtain payment or benefits from a healthcare program, such as Medicare or Medicaid, with the intention of personal gain. This can include falsifying claims, billing for services not rendered, or submitting false diagnoses or procedures to receive reimbursement.


In [6]:
def get_top_providers(metric="fraud_risk_score", limit=10, ascending=False):
    cols = [
        "rndrng_npi",
        "rndrng_prvdr_last_org_name",
        "rndrng_prvdr_first_name",
        "rndrng_prvdr_type",
        "rndrng_prvdr_state_abrvtn",
        "risk_category",
        "fraud_risk_score",
        "tot_mdcr_pymt_amt",
        "payment_per_beneficiary",
        "charge_to_payment_ratio",
        "services_per_beneficiary"
    ]

    return (
        df.sort_values(metric, ascending=ascending)
        [cols]
        .head(limit)
    )


def filter_providers(state=None, specialty=None, risk_category=None, limit=10):
    result = df.copy()

    if state:
        result = result[result["rndrng_prvdr_state_abrvtn"].str.upper() == state.upper()]

    if specialty:
        result = result[
            result["rndrng_prvdr_type"].str.contains(specialty, case=False, na=False)
        ]

    if risk_category:
        result = result[
            result["risk_category"].str.lower() == risk_category.lower()
        ]

    return (
        result.sort_values("fraud_risk_score", ascending=False)
        [
            [
                "rndrng_npi",
                "rndrng_prvdr_last_org_name",
                "rndrng_prvdr_first_name",
                "rndrng_prvdr_type",
                "rndrng_prvdr_state_abrvtn",
                "risk_category",
                "fraud_risk_score",
                "tot_mdcr_pymt_amt",
                "payment_per_beneficiary",
                "charge_to_payment_ratio"
            ]
        ]
        .head(limit)
    )


def summarize_dashboard_data():
    total_providers = df["rndrng_npi"].nunique()
    total_payments = df["tot_mdcr_pymt_amt"].sum()
    avg_risk = df["fraud_risk_score"].mean()
    high_risk_count = (df["risk_category"] == "High Risk").sum()

    top_state = (
        df[df["risk_category"] == "High Risk"]["rndrng_prvdr_state_abrvtn"]
        .value_counts()
        .idxmax()
    )

    top_specialty = (
        df.groupby("rndrng_prvdr_type")["fraud_risk_score"]
        .mean()
        .sort_values(ascending=False)
        .idxmax()
    )

    return f"""
Total Providers: {total_providers:,}
Total Medicare Payments: ${total_payments:,.2f}
Average Fraud Risk Score: {avg_risk:.2f}
High-Risk Providers: {high_risk_count:,}
Top High-Risk State: {top_state}
Highest Average Risk Specialty: {top_specialty}
"""

In [7]:
print("Top 10 Providers by Fraud Risk Score")
display(get_top_providers(metric="fraud_risk_score", limit=10))

print("Dashboard Summary")
print(summarize_dashboard_data())

Top 10 Providers by Fraud Risk Score


,rndrng_npi,rndrng_prvdr_last_org_name,rndrng_prvdr_first_name,rndrng_prvdr_type,rndrng_prvdr_state_abrvtn,risk_category,fraud_risk_score,tot_mdcr_pymt_amt,payment_per_beneficiary,charge_to_payment_ratio,services_per_beneficiary
1238979,1952581027,Hardesty,Brandon,Internal Medicine,IN,High Risk,100.000000,14649464.72,101030.791172,1.278498,44991.903448
850154,1659532927,Hedrick,David,Internal Medicine,IN,High Risk,100.000000,24725968.55,131521.109309,1.277765,66116.430851
37602,1023678240,Washington Institute For Coagulation,Unknown,Pharmacy,WA,High Risk,99.611729,8142471.12,226179.753333,1.275898,96669.722222
1193596,1922019561,Cascade Hemophilia Consortium,Unknown,Pharmacy,MI,High Risk,99.556337,13971685.25,465722.841667,1.297748,106727.433333
1185072,1912332289,Henning,Andrew,Nurse Practitioner,OR,High Risk,99.445609,5288253.88,151092.968000,1.342445,63334.914286
1139410,1871881573,Ethical Factor Rx Llc,Unknown,Pharmacy,PA,High Risk,99.445609,11542279.31,769485.287333,1.279297,419370.200000
499709,1386627966,Orthopaedic Hospital,Unknown,Pharmacy,CA,High Risk,99.390273,15830845.77,565387.348929,1.394959,166496.178571
762587,1588766968,Regents Of The University Of Colorado,Unknown,Pharmacy,CO,High Risk,99.169116,6831502.20,284645.925000,1.390147,45428.333333
713351,1548904360,Barton,Sherri,Nurse Practitioner,FL,High Risk,99.003444,6633882.13,34372.446269,1.567361,996.331606
904839,1699778563,Hemophilia Of Georgia,Unknown,Pharmacy,GA,High Risk,98.948258,9238531.10,219965.026190,1.277171,3780.428571


Dashboard Summary

Total Providers: 1,296,739
Total Medicare Payments: $120,057,897,699.85
Average Fraud Risk Score: 11.11
High-Risk Providers: 5,620
Top High-Risk State: CA
Highest Average Risk Specialty: Radiation Therapy Center



In [8]:
def build_context(question):
    q = question.lower()

    if "payment per beneficiary" in q:
        data = get_top_providers(metric="payment_per_beneficiary", limit=10)
        context_type = "Top providers by payment per beneficiary"

    elif "charge" in q or "charge-to-payment" in q:
        data = get_top_providers(metric="charge_to_payment_ratio", limit=10)
        context_type = "Top providers by charge-to-payment ratio"

    elif "medicare payment" in q or "highest payment" in q or "payments" in q:
        data = get_top_providers(metric="tot_mdcr_pymt_amt", limit=10)
        context_type = "Top providers by Medicare payments"

    elif "high risk" in q or "investigate" in q or "review first" in q:
        data = filter_providers(risk_category="High Risk", limit=10)
        context_type = "Top high-risk providers"

    elif "state" in q:
        data = (
            df.groupby("rndrng_prvdr_state_abrvtn")
            .agg(
                avg_fraud_risk=("fraud_risk_score", "mean"),
                high_risk_providers=("risk_category", lambda x: (x == "High Risk").sum()),
                total_medicare_payment=("tot_mdcr_pymt_amt", "sum"),
                total_providers=("rndrng_npi", "nunique")
            )
            .sort_values("high_risk_providers", ascending=False)
            .head(10)
        )
        context_type = "State-level fraud risk summary"

    elif "specialty" in q:
        data = (
            df.groupby("rndrng_prvdr_type")
            .agg(
                avg_fraud_risk=("fraud_risk_score", "mean"),
                high_risk_providers=("risk_category", lambda x: (x == "High Risk").sum()),
                total_medicare_payment=("tot_mdcr_pymt_amt", "sum"),
                total_providers=("rndrng_npi", "nunique")
            )
            .sort_values("avg_fraud_risk", ascending=False)
            .head(10)
        )
        context_type = "Specialty-level fraud risk summary"

    else:
        data = summarize_dashboard_data()
        context_type = "Dashboard overview"

    if isinstance(data, pd.DataFrame):
        data_text = data.to_string(index=False)
    else:
        data_text = str(data)

    return context_type, data_text

In [9]:
def build_prompt(question, context_type, data_context):
    prompt = f"""
You are an AI Healthcare Fraud Investigation Assistant.

Your role is to help healthcare fraud analysts interpret Medicare provider risk data.
Use only the provided data context. Do not invent provider names, numbers, or conclusions.

User Question:
{question}

Context Type:
{context_type}

Relevant Data:
{data_context}

Instructions:
- Answer in a clear fraud-investigation style.
- Do not claim fraud is proven.
- Use careful language such as "high-risk", "unusual billing pattern", "potential concern", or "recommended for review".
- Summarize the most important providers, states, specialties, or billing patterns.
- Explain why the result matters for healthcare fraud investigation.
- Keep the answer concise and practical.
"""
    return prompt

In [10]:
def ask_ai_investigator(question):
    context_type, data_context = build_context(question)
    prompt = build_prompt(question, context_type, data_context)
    answer = ask_llama(prompt)
    return answer

In [11]:
print(ask_ai_investigator("Which providers should investigators review first?"))

**Top High-Risk Providers Recommended for Review**

Based on the provided data, the following high-risk providers are recommended for review:

1. **Hardesty, Brandon (Internal Medicine)** - IN: With a high risk score of 100 and a total payment amount of $14,649,464.72, this provider's billing pattern is considered unusual.
2. **Hedrick, David (Internal Medicine)** - IN: Also with a high risk score of 100 and a total payment amount of $24,725,968.55, Hedrick's practice is flagged for potential concern.
3. **Cascade Hemophilia Consortium (Pharmacy)** - MI: This pharmacy has a high risk score of 99.56 and a total payment amount of $13,971,685.25, indicating possible overpayments.

These providers are recommended for review due to their:

* High risk scores
* Unusual billing patterns
* Large total payment amounts

The result matters because these providers may be engaging in practices that warrant further investigation to ensure compliance with Medicare regulations and prevent potential fi

In [12]:
print(ask_ai_investigator("Which providers have the highest Medicare payments?"))

**Top Providers by Medicare Payments: High-Risk Concerns**

Based on the provided data, we have identified the top providers with the highest Medicare payments. These providers may be exhibiting unusual billing patterns or high-risk behavior.

1. **Exact Sciences Laboratories, LLC**: Received $306,357,783 in Medicare payments, with a payment per beneficiary of $498.55 and a charge-to-payment ratio of 1.36.
2. Laboratory Corporation Of America Holdings (NC): Received $224,401,136 in Medicare payments, with a payment per beneficiary of $143.82 and a charge-to-payment ratio of 5.73.

Other high-paying providers include Guardant Health, Inc. (CA), Natera, Inc. (CA), and Quest Diagnostics Incorporated (NJ).

**Key Observations:**

* California (CA) has the highest number of high-risk providers with unusual billing patterns.
* Laboratory Corporation Of America Holdings (NC) and Quest Diagnostics Incorporated (NJ) both exhibit a high charge-to-payment ratio (> 5), which may indicate potential

In [13]:
print(ask_ai_investigator("Which specialties have the highest fraud risk?"))

**Fraud Investigation Insights**

Based on the provided data, we have identified potential concerns that warrant further review. The following highlights key findings:

* **Highest Average Risk Specialty:** Radiation Therapy Centers (Average Fraud Risk Score: 13.45)
This specialty has a significantly higher average fraud risk score compared to other specialties, indicating an unusual billing pattern or potential concern.

**High-Risk Providers:**

The top high-risk providers have been flagged for review due to their elevated fraud risk scores:

* **Top 5 High-Risk Specialties:** Radiation Therapy Centers (1,432), Cardiology (932), Dermatology (854)

These specialties account for a substantial portion of the total high-risk providers.

**Geographic Hotspots:**

The top high-risk state is California, with an average fraud risk score of 12.25.

**Implications for Healthcare Fraud Investigation:**

These findings suggest that Radiation Therapy Centers and Cardiology practices may be more s

In [ ]:
while True:

    question = input("\nAsk the Healthcare Fraud AI Assistant: ")

    if question.lower() in ["exit", "quit", "stop"]:
        print("\nHealthcare Fraud AI Assistant closed.")
        break

    try:
        answer = ask_ai_investigator(question)

        print("\n" + "="*80)
        print("Healthcare Fraud Investigation Report")
        print("="*80)
        print(answer)
        print("="*80 + "\n")

    except Exception as e:
        print(f"\nError: {e}\n")